# video-to-3d — phone video → 3D point cloud (Colab)

Modern, GPU path using **MASt3R** (DUSt3R's more accurate successor). No COLMAP, no provided camera calibration.

**Before you run:**
1. `Runtime → Change runtime type → Hardware accelerator: **T4 GPU**`
2. Edit `REPO_URL` in the setup cell to point at *your* fork.
3. `Runtime → Run all`, then upload a short clip when prompted.

Pipeline: **upload video → extract frames → blur filter → MASt3R → view → save .ply**

## 0. Check the GPU
If this prints nothing, you forgot to switch the runtime to a T4 GPU.

In [ ]:
!nvidia-smi -L

## 1. Setup — clone the repo + MASt3R, install deps
Edit `REPO_URL` to your own GitHub fork (that's how Colab gets the `v3d/` code).
MASt3R is cloned with its DUSt3R submodule, which it imports as a dependency.

In [ ]:
REPO_URL = "https://github.com/maheswariridhi/video-to-3d.git"  # <-- change if your repo differs

import os, sys
if not os.path.isdir("video-to-3d"):
    !git clone $REPO_URL
%cd video-to-3d
!pip install -q -r requirements.txt

# MASt3R — the reconstruction engine (clones DUSt3R as a submodule).
if not os.path.isdir("mast3r"):
    !git clone --recursive https://github.com/naver/mast3r
    !pip install -q -r mast3r/requirements.txt
    !pip install -q -r mast3r/dust3r/requirements.txt
sys.path += ["mast3r", "mast3r/dust3r"]

## 2. Upload your video
A 10–30 s slow sweep of a small room works best. (Or mount Google Drive instead.)

In [ ]:
from google.colab import files
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print("uploaded:", video_path)

## 3. Prep frames (CPU): extract → deblur → thin to fit GPU memory

In [ ]:
from v3d import frames

frames.extract(video_path, "out/images", fps=2)
frames.filter_blurry("out/images")
frames.subsample("out/images", max_frames=25)  # denser coverage; stays on DUSt3R's "complete" graph (<=25)

## 4. Reconstruct (GPU): MASt3R → coloured point cloud
First run downloads the model (~2 GB).

In [ ]:
from v3d import reconstruct

# Raise min_conf (e.g. 3, 5) for fewer but cleaner points.
rec = reconstruct.run("out/images", device="cuda")

## 5. Semantic labels in 3D (optional)
Segment each frame (SegFormer / ADE20K), lift the labels onto the 3D points
(so they're aligned with the geometry), then voxel-majority-vote for multi-view
consistency. Furniture classes are coloured; everything else stays grey.

## 6. View inline + save outputs
Renders the cloud here and writes `point_cloud.ply`, a standalone `preview.html`,
and `report.json` into `out/` (download them from the Files panel when needed).

import json
from pathlib import Path
from v3d import pointcloud

# Show the cloud inline + save a standalone HTML you can reopen in any browser.
fig = pointcloud.show(rec.points, rec.colors)
fig.write_html("out/preview.html")
pointcloud.save_ply(rec.points, rec.colors, "out/point_cloud.ply")

report = {
    "num_frames_used": len(list(Path("out/images").glob("frame_*.jpg"))),
    "num_points": int(len(rec.points)),
    "outputs": sorted(str(p) for p in Path("out").glob("*") if p.is_file()),
}
Path("out/report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
print("\nSaved under out/. Grab any file from the Files panel (📁 on the left) when you want it —")
print("e.g. right-click point_cloud.ply → Download.")

In [ ]:
import json
import shutil
from pathlib import Path

from google.colab import files
from v3d import pointcloud

# Interactive view inline + a standalone HTML a reviewer can open without Colab.
fig = pointcloud.show(rec.points, rec.colors)
fig.write_html("out/preview.html")

pointcloud.save_ply(rec.points, rec.colors, "out/point_cloud.ply")

report = {
    "num_frames_used": len(list(Path("out/images").glob("frame_*.jpg"))),
    "num_points": int(len(rec.points)),
    "output_ply": "out/point_cloud.ply",
    "preview_html": "out/preview.html",
}
Path("out/report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

# Bundle everything (ply + preview.html + report.json + images/) into one download.
shutil.make_archive("video_to_3d_outputs", "zip", "out")
files.download("video_to_3d_outputs.zip")